In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os
import pandas as pd
from tqdm import tqdm

# Allow more columns to be displayed
pd.set_option("display.max_columns", None)

import logging
logging.basicConfig(level=logging.WARNING)

In [13]:
# Allow imports from parent directory
import sys
sys.path.append('..')
from utils.flood_request_utils import (
    # Hazard
    get_wri_and_si_hazard_data,
    plot_wri_and_si_hazard_data,

    # Vulnerability
    plot_wri_and_si_vulnerability_data,
    apply_damage_fraction,
    get_damage_fraction
)

In [14]:
!which python

/Users/klemenkubelj/miniconda3/envs/cvar-masters/bin/python


In [15]:
DATA_DIR = "../data"

In [76]:
# Define the column names we want to check
predicted_columns = [
    "predicted_wri_depth",
    "predicted_si_depth",
    "predicted_si_old_depth",
    "predicted_wri_damage",
    "predicted_si_damage",
    "predicted_si_old_damage"
]

In [77]:
vloge_df = pd.read_csv(os.path.join(DATA_DIR, "vloge_processed_with_gps_filtered_2025-05-10.csv"))
vloge_df.shape

(13531, 18)

In [78]:
# NOTE: Comment out this if you want to calculate the flood depths from scratch
cached_df = pd.read_excel(os.path.join("../data/predicted_flood_depths_2025-05-13.xlsx"))
vloge_df = vloge_df.merge(cached_df[
    ["VlogaId"] + predicted_columns
], on=["VlogaId"], how="left")

In [79]:
# vloge_df.loc[10:15, predicted_columns] = pd.NA
# vloge_df[vloge_df["predicted_wri_depth"].isna()]

In [80]:
def calc_predicted_flood_depth(row):
    # Get the hazard data
    gps = {
        "lat": row["gps_lat"],
        "lng": row["gps_lng"]
    }
    data, request = get_wri_and_si_hazard_data(gps)
    return (
        data["flood_depths"]["wri"][100],
        data["flood_depths"]["si"][100],
        data["flood_depths"]["si_old"][100],
        data["damage_fractions"]["wri"][100],
        data["damage_fractions"]["si"][100],
        data["damage_fractions"]["si_old"][100]
    )

_df = vloge_df.copy()



# Create the columns if they don't exist
for col in predicted_columns:
    if col not in _df.columns:
        _df[col] = None

# Create a mask for rows that need calculation
needs_calculation = _df[predicted_columns].isna().any(axis=1)

# Only process rows that need calculation
if needs_calculation.any():
    # Create a tqdm progress bar for the DataFrame rows
    tqdm.pandas(desc="Processing rows")
    # Apply the function with progress bar only to rows that need calculation
    results = _df[needs_calculation].progress_apply(calc_predicted_flood_depth, axis=1)
    
    # Update only the rows that needed calculation
    _df.loc[needs_calculation, predicted_columns] = list(zip(*results))

## Export

In [82]:
_df["measured_depth"] = _df["Objekt_VisinaVodeCm"]/100

# Export all cols but "gps_point"
_df.to_excel(os.path.join("../data/predicted_flood_depths_2025-05-13.xlsx"), index=False)